# GradTree Project: Preprocessing for Spambase & Wine (Myat)

Pipeline: **load → inspect → stratified split (80/20, then 20% of train as validation) → quantile transform (fit on train only) → optional SMOTE (train only) → label-encode → save**.

## 1. Setup

In [1]:
!pip install -q ucimlrepo imbalanced-learn

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import SMOTE

In [3]:
# Save to Google Drive so files persist. Set USE_DRIVE = False to save inside the Colab runtime instead.
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = Path('/content/drive/MyDrive/gradtree-project/data/processed')
else:
    OUT_DIR = Path('data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Saving to:', OUT_DIR)

Mounted at /content/drive
Saving to: /content/drive/MyDrive/gradtree-project/data/processed


## 2. Settings

All datasets should be processed the same way.

In [5]:
SEEDS = range(10)          # one split per seed, for the 10 runs
TEST_SIZE = 0.2            # 80/20 train/test
VAL_SIZE = 0.2             # 20% of the training data held out for early stopping
DROP_DUPLICATES = True     # Spambase has duplicate rows; keeping them lets copies land in both train and test
SMOTE_THRESHOLD = 0.25     # apply SMOTE only if the smallest class is below this share
QT_OUTPUT = 'normal'       # check the official GradTree repo and match its output_distribution

## 3. Load the data from UCI

In [6]:
spam = fetch_ucirepo(id=94)
X_spam = spam.data.features
y_spam = spam.data.targets.squeeze()

wine = fetch_ucirepo(id=109)
X_wine = wine.data.features
y_wine = wine.data.targets.squeeze()

DATASETS = {'spambase': (X_spam, y_spam), 'wine': (X_wine, y_wine)}

## 4. Inspect before changing anything

Expected: Spambase 4,601 × 57, Wine 178 × 13, no missing values, no categorical columns.

In [7]:
for name, (X, y) in DATASETS.items():
    print(f'=== {name} ===')
    print('Shape:', X.shape)
    print('Missing values:', int(X.isna().sum().sum()), '| missing targets:', int(y.isna().sum()))
    print('Non-numeric columns:', list(X.select_dtypes(exclude='number').columns))
    print('Duplicate rows (features + label):', int(pd.concat([X, y], axis=1).duplicated().sum()))
    print('Class balance:')
    print(y.value_counts(normalize=True).round(3).to_string())
    print()

=== spambase ===
Shape: (4601, 57)
Missing values: 0 | missing targets: 0
Non-numeric columns: []
Duplicate rows (features + label): 391
Class balance:
Class
0    0.606
1    0.394

=== wine ===
Shape: (178, 13)
Missing values: 0 | missing targets: 0
Non-numeric columns: []
Duplicate rows (features + label): 0
Class balance:
class
2    0.399
1    0.331
3    0.270



In [8]:
X_wine.describe().T  # change to X_spam to look at Spambase

,count,mean,std,min,25%,50%,75%,max
Alcohol,178.0,13.000618,0.811827,11.03,12.3625,13.050,13.6775,14.83
Malicacid,178.0,2.336348,1.117146,0.74,1.6025,1.865,3.0825,5.80
Ash,178.0,2.366517,0.274344,1.36,2.2100,2.360,2.5575,3.23
Alcalinity_of_ash,178.0,19.494944,3.339564,10.60,17.2000,19.500,21.5000,30.00
Magnesium,178.0,99.741573,14.282484,70.00,88.0000,98.000,107.0000,162.00
Total_phenols,178.0,2.295112,0.625851,0.98,1.7425,2.355,2.8000,3.88
Flavanoids,178.0,2.029270,0.998859,0.34,1.2050,2.135,2.8750,5.08
Nonflavanoid_phenols,178.0,0.361854,0.124453,0.13,0.2700,0.340,0.4375,0.66
Proanthocyanins,178.0,1.590899,0.572359,0.41,1.2500,1.555,1.9500,3.58
Color_intensity,178.0,5.058090,2.318286,1.28,3.2200,4.690,6.2000,13.00


## 5. Clean

In [9]:
def clean(X, y):
    X, y = X.copy(), y.copy()
    if DROP_DUPLICATES:
        keep = ~pd.concat([X, y], axis=1).duplicated()
        X, y = X[keep], y[keep]
    return X.reset_index(drop=True), y.reset_index(drop=True)

for name, (X, y) in DATASETS.items():
    Xc, yc = clean(X, y)
    DATASETS[name] = (Xc, yc)
    print(f'{name}: {X.shape[0]} -> {Xc.shape[0]} rows after cleaning')

spambase: 4601 -> 4210 rows after cleaning
wine: 178 -> 178 rows after cleaning


## 6. Shared pipeline: split → transform → SMOTE → encode

Every transform is **fit on the training set only**, then applied to validation and test. This is the shape of function to move into `src/preprocessing/common.py` later so the whole team uses it.

In [10]:
def finish_pipeline(X, y, seed):
    X = X.to_numpy(dtype=float)

    # Encode labels to 0..K-1 (Wine uses 1/2/3; PyTorch cross-entropy needs 0-based)
    le = LabelEncoder().fit(y)
    y = le.transform(y)

    # Stratified 80/20, then 20% of train for validation
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=seed)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tr, y_tr, test_size=VAL_SIZE, stratify=y_tr, random_state=seed)

    # Quantile transform: fit on train only. n_quantiles is capped for small datasets like Wine.
    qt = QuantileTransformer(n_quantiles=min(1000, len(X_tr)),
                             output_distribution=QT_OUTPUT, random_state=seed)
    X_tr = qt.fit_transform(X_tr)
    X_val = qt.transform(X_val)
    X_te = qt.transform(X_te)

    # SMOTE on the training set only, and only if imbalanced
    minority_share = np.bincount(y_tr).min() / len(y_tr)
    smote_used = minority_share < SMOTE_THRESHOLD
    if smote_used:
        X_tr, y_tr = SMOTE(random_state=seed).fit_resample(X_tr, y_tr)

    return dict(X_train=X_tr, X_val=X_val, X_test=X_te,
                y_train=y_tr, y_val=y_val, y_test=y_te,
                classes=le.classes_.astype(str), smote_used=smote_used)

## 7. Run for all seeds and save

In [11]:
feature_names = {}
for name, (X, y) in DATASETS.items():
    feature_names[name] = list(X.columns)
    for seed in SEEDS:
        d = finish_pipeline(X, y, seed)
        path = OUT_DIR / name / f'seed_{seed}.npz'
        path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(path, **d, feature_names=np.array(feature_names[name]))
    print(f'{name}: saved {len(SEEDS)} seeds | train {d["X_train"].shape}, '
          f'val {d["X_val"].shape}, test {d["X_test"].shape} | SMOTE: {d["smote_used"]}')

spambase: saved 10 seeds | train (2694, 57), val (674, 57), test (842, 57) | SMOTE: False
wine: saved 10 seeds | train (113, 13), val (29, 13), test (36, 13) | SMOTE: False


## 8. Sanity checks

1. Class proportions should match across train/val/test.
2. Transformed training features should look roughly normal (mean ≈ 0, std ≈ 1 if `QT_OUTPUT='normal'`).
3. A quick CART should get roughly 90% accuracy on Spambase and above 85% on Wine. Much worse usually means a pipeline bug.

In [12]:
def load_split(name, seed):
    d = np.load(OUT_DIR / name / f'seed_{seed}.npz', allow_pickle=True)
    return {k: d[k] for k in d.files}

for name in DATASETS:
    d = load_split(name, 0)
    print(f'=== {name} (seed 0) ===')
    for part in ['train', 'val', 'test']:
        props = np.bincount(d[f'y_{part}']) / len(d[f'y_{part}'])
        print(f'  {part:5s} class shares: {np.round(props, 3)}')
    print(f'  train feature mean {d["X_train"].mean():.2f}, std {d["X_train"].std():.2f}')

    accs, f1s = [], []
    for seed in SEEDS:
        d = load_split(name, seed)
        pred = DecisionTreeClassifier(random_state=seed).fit(d['X_train'], d['y_train']).predict(d['X_test'])
        accs.append(accuracy_score(d['y_test'], pred))
        f1s.append(f1_score(d['y_test'], pred, average='macro'))
    print(f'  CART over {len(SEEDS)} seeds: accuracy {np.mean(accs):.3f} ± {np.std(accs):.3f}, '
          f'macro F1 {np.mean(f1s):.3f} ± {np.std(f1s):.3f}\n')

=== spambase (seed 0) ===
  train class shares: [0.601 0.399]
  val   class shares: [0.601 0.399]
  test  class shares: [0.601 0.399]
  train feature mean -3.79, std 2.62
  CART over 10 seeds: accuracy 0.900 ± 0.016, macro F1 0.896 ± 0.017

=== wine (seed 0) ===
  train class shares: [0.336 0.398 0.265]
  val   class shares: [0.31  0.414 0.276]
  test  class shares: [0.333 0.389 0.278]
  train feature mean -0.00, std 1.19
  CART over 10 seeds: accuracy 0.906 ± 0.073, macro F1 0.904 ± 0.074



## Loading the splits later

In the GradTree or CART training code:
```python
d = np.load('.../data/processed/wine/seed_0.npz', allow_pickle=True)
X_train, y_train = d['X_train'], d['y_train']
```